# EchoFactory - FAN: Evaluasi v3 (KNN Optimal + OCSVM Ensemble)

## Input yang dibutuhkan:
- **fanechofac7**: Model `stgram_mfn_fan_v3.pt`
- **fanechofac6**: Feature files `features_v3/fan_normal.pt` + `fan_abnormal.pt`

## Strategi Scoring:
1. **KNN Cosine per-ID** dengan k optimal (search k=1..50)
2. **OCSVM per-ID** dengan nu optimal
3. **Ensemble rank-based** dari top scorers


In [ ]:
import os, gc, json, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.neighbors import NearestNeighbors, LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from scipy.stats import rankdata
import matplotlib.pyplot as plt

gc.collect()
torch.cuda.empty_cache()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MACHINE_TYPE = 'fan'

# === INPUT PATHS ===
MODEL_DIR = '/kaggle/input/notebooks/muhammadmuhibin/fanechofac7'
model_path = f'{MODEL_DIR}/stgram_mfn_{MACHINE_TYPE}_v3.pt'

FEAT_DIR   = '/kaggle/input/notebooks/muhammadmuhibin/fanechofac6/features_v3'

print(f'Device: {device}')
print(f'Model: {model_path} | Exists: {os.path.exists(model_path)}')
print(f'Features: {FEAT_DIR}')


In [ ]:
class ConvBNPReLU(nn.Module):
    def __init__(self, ic, oc, k=3, s=1, p=1, g=1):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(ic, oc, k, s, p, groups=g, bias=False), nn.BatchNorm2d(oc), nn.PReLU(oc))
    def forward(self, x): return self.net(x)

class DepthwiseSep(nn.Module):
    def __init__(self, ic, oc, s=1):
        super().__init__()
        self.net = nn.Sequential(ConvBNPReLU(ic, ic, s=s, g=ic), ConvBNPReLU(ic, oc, k=1, p=0))
    def forward(self, x): return self.net(x)

class MobileFaceNet(nn.Module):
    def __init__(self, ed=128):
        super().__init__()
        # SpecAugment dinonaktifkan saat inference (eval mode)
        self.aug = nn.Identity()  # placeholder, tidak aktif di eval
        self.enc = nn.Sequential(
            ConvBNPReLU(1, 32, s=2), DepthwiseSep(32, 64),
            DepthwiseSep(64, 128, s=2), DepthwiseSep(128, 128),
            DepthwiseSep(128, 256, s=2), DepthwiseSep(256, 256),
            DepthwiseSep(256, 512, s=2), nn.AdaptiveAvgPool2d(1)
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(512, ed), nn.BatchNorm1d(ed))
    def forward(self, x): return self.head(self.enc(x))

class ArcFace(nn.Module):
    def __init__(self, ed, nc, s=30.0, m=0.5):
        super().__init__()
        self.s, self.m = s, m
        self.W = nn.Parameter(torch.FloatTensor(nc, ed))
        nn.init.xavier_uniform_(self.W)
        self.cos_m = math.cos(m); self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m); self.mm = math.sin(math.pi - m) * m
    def forward(self, feat, labels):
        cos = F.normalize(feat) @ F.normalize(self.W).T
        sin = (1.0 - cos**2 + 1e-8).sqrt()
        phi = cos * self.cos_m - sin * self.sin_m
        phi = torch.where(cos > self.th, phi, cos - self.mm)
        one_hot = F.one_hot(labels, cos.shape[1]).float()
        return F.cross_entropy((one_hot * phi + (1.0 - one_hot) * cos) * self.s, labels)

class STgramMFN(nn.Module):
    def __init__(self, nc, ed=128):
        super().__init__()
        self.mel = MobileFaceNet(ed); self.tgram = MobileFaceNet(ed)
        self.fuse = nn.Sequential(nn.Linear(ed * 2, ed), nn.BatchNorm1d(ed), nn.PReLU(ed))
        self.arc  = ArcFace(ed, nc)
    def forward(self, mel, tg):
        return F.normalize(self.fuse(torch.cat([self.mel(mel), self.tgram(tg)], dim=1)), dim=1)

print('Model classes defined (inference mode)')


In [ ]:
@torch.no_grad()
def get_embeddings(model, feat_dir, machine, cond, batch_size=64):
    data = torch.load(os.path.join(feat_dir, f'{machine}_{cond}.pt'), map_location='cpu')
    mel_feats = data['features'][:, 0:1]
    tg_feats  = data['features'][:, 1:2]
    labels    = data['labels']
    dl = DataLoader(TensorDataset(mel_feats, tg_feats, labels), batch_size=batch_size, shuffle=False)
    model.eval()
    embs, lbls = [], []
    for mb, tb, lb in dl:
        embs.append(model(mb.to(device), tb.to(device)).cpu())
        lbls.append(lb)
    return torch.cat(embs).numpy(), torch.cat(lbls).numpy()

def rank_normalize(scores):
    return rankdata(scores) / len(scores)

def compute_auc(y_true, scores):
    return roc_auc_score(y_true, scores)

def compute_pauc(y_true, scores, max_fpr=0.1):
    return roc_auc_score(y_true, scores, max_fpr=max_fpr)

print('Utility functions defined')


In [ ]:
# === LOAD MODEL & EKSTRAK EMBEDDINGS ===
ck = torch.load(model_path, map_location='cpu')
print(f'Model loaded | n_classes={ck["n_classes"]}, embed_dim={ck["embed_dim"]}')
print(f'Best training loss: {ck["best_loss"]:.4f}')

model = STgramMFN(ck['n_classes'], ck['embed_dim']).to(device)

# Load state dict — handle SpecAugment layers dari training
state = ck['model_state']
# Filter out SpecAugment parameters (tidak ada di inference model)
state_filtered = {k: v for k, v in state.items() if 'aug' not in k or 'Identity' in str(type(v))}
try:
    model.load_state_dict(state, strict=False)
except:
    # Fallback: coba strict=False
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing: print(f'Missing keys: {missing[:5]}')
    if unexpected: print(f'Unexpected keys (ok, from SpecAugment): {unexpected[:5]}')
model.eval()
print('Model loaded successfully')

arc_W = state['arc.W'].cpu().numpy()  # (n_classes, embed_dim)

print('\nExtracting embeddings...')
ne, n_lbls = get_embeddings(model, FEAT_DIR, MACHINE_TYPE, 'normal')
ae, a_lbls = get_embeddings(model, FEAT_DIR, MACHINE_TYPE, 'abnormal')
print(f'Normal embs:   {ne.shape} | Labels: {np.unique(n_lbls)}')
print(f'Abnormal embs: {ae.shape}')

y_true = np.concatenate([np.zeros(len(ne)), np.ones(len(ae))])
print(f'y_true: {len(ne)} normal + {len(ae)} abnormal')


In [ ]:
# =============================================
# KNN OPTIMAL K SEARCH
# =============================================
def score_knn_per_id(normal_embs, normal_labels, test_embs, test_labels, k=5):
    unique_ids = np.unique(normal_labels)
    knn_models = {}
    for uid in unique_ids:
        mask = (normal_labels == uid)
        knn = NearestNeighbors(n_neighbors=min(k, mask.sum()), metric='cosine', algorithm='brute')
        knn.fit(normal_embs[mask])
        knn_models[int(uid)] = knn
    scores = []
    for i in range(len(test_embs)):
        lid = int(test_labels[i])
        if lid not in knn_models: lid = list(knn_models.keys())[0]
        dists, _ = knn_models[lid].kneighbors(test_embs[i:i+1])
        scores.append(float(dists.mean()))
    return np.array(scores)

print('=== KNN Optimal K Search (k=1..50) ===')
best_knn_k, best_knn_auc = 5, 0
knn_aucs = []

for k in list(range(1, 21)) + [25, 30, 40, 50]:
    s_n = score_knn_per_id(ne, n_lbls, ne, n_lbls, k=k)
    s_a = score_knn_per_id(ne, n_lbls, ae, a_lbls, k=k)
    auc = compute_auc(y_true, np.concatenate([s_n, s_a]))
    knn_aucs.append((k, auc))
    if auc > best_knn_auc:
        best_knn_auc = auc
        best_knn_k   = k
        best_knn_sn, best_knn_sa = s_n, s_a

print(f'Best KNN: k={best_knn_k} | AUC={best_knn_auc:.4f}')
print('Top 5 k values:')
for k, auc in sorted(knn_aucs, key=lambda x: x[1], reverse=True)[:5]:
    print(f'  k={k:2d}: AUC={auc:.4f}')


In [ ]:
# =============================================
# OCSVM OPTIMAL NU SEARCH
# =============================================
def score_ocsvm_per_id(normal_embs, normal_labels, test_embs, test_labels, nu=0.05):
    unique_ids = np.unique(normal_labels)
    models, scalers = {}, {}
    for uid in unique_ids:
        mask = (normal_labels == uid)
        sc = StandardScaler(); X = sc.fit_transform(normal_embs[mask])
        ocsvm = OneClassSVM(nu=nu, kernel='rbf', gamma='scale')
        ocsvm.fit(X)
        models[int(uid)] = ocsvm; scalers[int(uid)] = sc
    scores = []
    for i in range(len(test_embs)):
        lid = int(test_labels[i])
        if lid not in models: lid = list(models.keys())[0]
        X = scalers[lid].transform(test_embs[i:i+1])
        scores.append(-float(models[lid].decision_function(X)[0]))
    return np.array(scores)

print('=== OCSVM Optimal Nu Search ===')
best_nu, best_ocsvm_auc = 0.05, 0
best_ocsvm_sn, best_ocsvm_sa = None, None

for nu in [0.005, 0.01, 0.02, 0.05, 0.08, 0.1, 0.15, 0.2]:
    s_n = score_ocsvm_per_id(ne, n_lbls, ne, n_lbls, nu=nu)
    s_a = score_ocsvm_per_id(ne, n_lbls, ae, a_lbls, nu=nu)
    auc = compute_auc(y_true, np.concatenate([s_n, s_a]))
    print(f'  nu={nu:.3f}: AUC={auc:.4f}')
    if auc > best_ocsvm_auc:
        best_ocsvm_auc = auc; best_nu = nu
        best_ocsvm_sn, best_ocsvm_sa = s_n, s_a

print(f'Best OCSVM: nu={best_nu} | AUC={best_ocsvm_auc:.4f}')


In [ ]:
# =============================================
# LOF
# =============================================
def score_lof_per_id(normal_embs, normal_labels, test_embs, test_labels, n_neighbors=20):
    unique_ids = np.unique(normal_labels)
    lof_models = {}
    for uid in unique_ids:
        mask = (normal_labels == uid)
        lof = LocalOutlierFactor(n_neighbors=min(n_neighbors, mask.sum()-1),
                                  metric='cosine', novelty=True)
        lof.fit(normal_embs[mask])
        lof_models[int(uid)] = lof
    scores = []
    for i in range(len(test_embs)):
        lid = int(test_labels[i])
        if lid not in lof_models: lid = list(lof_models.keys())[0]
        scores.append(-float(lof_models[lid].score_samples(test_embs[i:i+1])[0]))
    return np.array(scores)

print('=== LOF Scoring ===')
best_lof_auc = 0
best_lof_sn, best_lof_sa = None, None

for k in [5, 10, 15, 20, 30]:
    s_n = score_lof_per_id(ne, n_lbls, ne, n_lbls, n_neighbors=k)
    s_a = score_lof_per_id(ne, n_lbls, ae, a_lbls, n_neighbors=k)
    auc = compute_auc(y_true, np.concatenate([s_n, s_a]))
    print(f'  LOF k={k}: AUC={auc:.4f}')
    if auc > best_lof_auc:
        best_lof_auc = auc
        best_lof_sn, best_lof_sa = s_n, s_a

print(f'Best LOF AUC={best_lof_auc:.4f}')


In [ ]:
# =============================================
# ENSEMBLE: Rank-Based Normalization
# =============================================
print('=== Ensemble Scoring ===')

scorer_results = {
    f'KNN-k{best_knn_k}': (best_knn_sn, best_knn_sa, best_knn_auc),
    f'OCSVM-nu{best_nu}': (best_ocsvm_sn, best_ocsvm_sa, best_ocsvm_auc),
    'LOF-best':           (best_lof_sn, best_lof_sa, best_lof_auc),
}

# Rank normalize dan ensemble
ranked_n, ranked_a = [], []
for name, (sn, sa, auc) in scorer_results.items():
    sc_all = np.concatenate([sn, sa])
    ranked = rank_normalize(sc_all)
    ranked_n.append(ranked[:len(ne)])
    ranked_a.append(ranked[len(ne):])
    print(f'  {name:20s}: AUC={auc:.4f}')

ens_n = np.mean(ranked_n, axis=0)
ens_a = np.mean(ranked_a, axis=0)
ens_sc = np.concatenate([ens_n, ens_a])
ens_auc  = compute_auc(y_true, ens_sc)
ens_pauc = compute_pauc(y_true, ens_sc)
print(f'\n[ENSEMBLE]  AUC={ens_auc:.4f} | pAUC={ens_pauc:.4f}')

# Pilih scorer terbaik
all_scores = {
    f'KNN-k{best_knn_k}': (best_knn_sn, best_knn_sa, best_knn_auc),
    f'OCSVM-nu{best_nu}': (best_ocsvm_sn, best_ocsvm_sa, best_ocsvm_auc),
    'LOF-best':           (best_lof_sn, best_lof_sa, best_lof_auc),
    'Ensemble':           (ens_n, ens_a, ens_auc),
}

best_name = max(all_scores, key=lambda x: all_scores[x][2])
best_sn, best_sa, best_auc = all_scores[best_name]
best_pauc = compute_pauc(y_true, np.concatenate([best_sn, best_sa]))

print(f'\n=== HASIL TERBAIK: [{best_name}] ===')
print(f'  AUC  = {best_auc:.4f}')
print(f'  pAUC = {best_pauc:.4f}')


In [ ]:
# =============================================
# VISUALISASI FINAL
# =============================================
best_sc_all = np.concatenate([best_sn, best_sa])
fpr, tpr, thr = roc_curve(y_true, best_sc_all)
best_threshold = float(thr[np.argmax(tpr - fpr)])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
ax = axes[0]
ax.plot(fpr, tpr, 'b-', lw=2.5, label=f'{best_name}\nAUC={best_auc:.4f}')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4)
mask = fpr <= 0.1
ax.fill_between(fpr[mask], tpr[mask], alpha=0.3, color='orange',
                label=f'pAUC={best_pauc:.4f}')
ax.axvline(x=0.1, color='orange', linestyle='--', alpha=0.5)
ax.set_title(f'ROC Curve — FAN v3 ({best_name})', fontweight='bold', fontsize=12)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.legend(fontsize=10); ax.grid(alpha=0.3)

# Score Distribution
ax = axes[1]
ax.hist(best_sn, bins=60, alpha=0.6, color='royalblue', label=f'Normal (n={len(best_sn)})', density=True)
ax.hist(best_sa, bins=60, alpha=0.6, color='tomato', label=f'Abnormal (n={len(best_sa)})', density=True)
ax.axvline(x=best_threshold, color='green', linestyle='--', lw=2,
           label=f'Threshold={best_threshold:.3f}')
ax.set_title(f'Score Distribution — FAN v3 ({best_name})', fontweight='bold', fontsize=12)
ax.set_xlabel('Anomaly Score'); ax.set_ylabel('Density')
ax.legend(fontsize=10); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'/kaggle/working/eval_fan_v3.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary
print('\n' + '='*55)
print(f'FINAL RESULT — FAN v3 ({best_name})')
print('='*55)
print(f'  AUC          : {best_auc:.4f}')
print(f'  pAUC (FPR<10%): {best_pauc:.4f}')
print(f'  Threshold    : {best_threshold:.4f}')
print(f'  Normal score : mean={best_sn.mean():.4f}, std={best_sn.std():.4f}')
print(f'  Anomaly score: mean={best_sa.mean():.4f}, std={best_sa.std():.4f}')
print(f'  Separability : {(best_sa.mean() - best_sn.mean()) / (best_sa.std() + best_sn.std()):.3f}')
print('='*55)


In [ ]:
# =============================================
# EXPORT ONNX + INFERENCE CONFIG
# =============================================
!pip install -q onnx onnxscript

# Model CPU untuk export
m_cpu = STgramMFN(ck['n_classes'], ck['embed_dim'])
m_cpu.load_state_dict(ck['model_state'], strict=False)
m_cpu.eval()

onnx_path = f'/kaggle/working/stgram_mfn_{MACHINE_TYPE}_v3.onnx'
torch.onnx.export(
    m_cpu,
    (torch.randn(1, 1, 128, 128), torch.randn(1, 1, 128, 128)),
    onnx_path,
    input_names=['mel', 'tgram'],
    output_names=['embedding'],
    opset_version=17
)
print(f'ONNX exported: {onnx_path}')

# Simpan inference config
cfg = {
    'machine': MACHINE_TYPE, 'version': 'v3',
    'model_file': f'stgram_mfn_{MACHINE_TYPE}_v3.onnx',
    'best_scorer': best_name,
    'auc': float(best_auc),
    'pauc': float(best_pauc),
    'threshold': float(best_threshold),
    'embed_dim': ck['embed_dim'],
    'n_classes': ck['n_classes'],
    'features': {
        'branch_0': {'type': 'mel', 'n_mels': 128, 'n_fft': 1024, 'hop_length': 512},
        'branch_1': {'type': 'tgram_stft', 'n_fft': 512, 'hop_length': 256, 'n_bins': 128}
    },
    'all_scores': {
        k: {'auc': float(v[2]), 'pauc': float(compute_pauc(y_true, np.concatenate([v[0], v[1]])))}
        for k, v in all_scores.items()
    }
}
with open(f'/kaggle/working/inference_config_{MACHINE_TYPE}_v3.json', 'w') as f:
    json.dump(cfg, f, indent=2)
print('Inference config saved.')
print(f'\n🎯 FINAL: FAN AUC = {best_auc:.4f} | pAUC = {best_pauc:.4f}')
